Пишем код для учета атмосферной рефракции в приближении плоско-параллельной атмосферы

In [1]:
from math import *
import numpy as np
from datetime import datetime
from astropy.time import Time

def tangFromRADE(ra, dec, RA, DEC):
    ksi = cos(dec)*sin(ra-RA)/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    eta = (sin(dec)*cos(DEC)-cos(dec)*sin(DEC)*cos(ra-RA))/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    return ksi,eta

def RADecFromTang(ksi, eta, RA, Dec):
    x,y,z = np.dot(np.array([[-sin(RA),-cos(RA) * sin(Dec),cos(RA) * cos(Dec)],
                             [cos(RA),-sin(RA) * sin(Dec),sin(RA) * cos(Dec)],
                             [0,cos(Dec),sin(Dec)]]),
                   np.array([ksi,eta,1]))/sqrt(1+ksi*ksi+eta*eta)
    ra = atan2(y,x)
    dec = atan2(z,sqrt(x*x+y*y))
    if(ra<0):
        ra+=2*pi
    return ra,dec
# это функция для вычисления звездного времени
def getSiderial(JD, lon):
    MJD = JD - 2400000.5
    H = (MJD - trunc(MJD))*24.0
    D = JD - 2451545.0
    D0 = trunc(MJD) - 51544.5
    T = D/36525
    LST = 6.697374558 + 0.06570982441908*D0 + 1.00273790935*H + 0.000026*T*T+degrees(lon)/15
    return pi*(LST - 24.0*trunc(LST/24.0))/12.0

def RADECtoAzEl(ra,dec,s,lat):
    Phi = lat-pi/2.0
    e = np.array([cos(s-ra)*cos(dec),sin(s-ra)*cos(dec),sin(dec)])
    h = np.array([cos(Phi)*e[0]+sin(Phi)*e[2],e[1],-sin(Phi)*e[0]+cos(Phi)*e[2]])
    cosd = sqrt(h[0] * h[0] + h[1] * h[1])
    return atan2(h[1] / cosd, h[0] / cosd), atan2(h[2], cosd)

In [2]:
def refraction(ra,dec, s, lat, JD, t, P, wl):
    ra_ref,dec_ref = ra,dec
    return ra_ref,dec_ref


In [3]:
lon = radians(30.327498)
lat = radians(59.771831)
P = 750
t = 7
wl = 0.55
ra,dec = radians(60.0),radians(-20.0)

JD = float(Time(datetime.utcnow()).copy(format='jd').value)
print(JD)

2461171.109326088


/var/folders/hy/8vvv51316x9388z2zyw61jgm0000gn/T/ipykernel_23646/2432350929.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  JD = float(Time(datetime.utcnow()).copy(format='jd').value)


In [4]:
# просто потренируйтесь вычислять координаты, исправленные за влияние атмосферной рефракции
ra,dec = 30,30
s = getSiderial(JD,lon)
ra_ref, dec_ref = refraction(ra,dec,s,lat,JD,t,P,wl)

Допустим звезда имеет координаты ra,dec = s,30 (s - звездное время) в начальный момент наблюдений. Околоземный астероид проходит вблизи звезды и его координаты меняются линейно со временем. В момент s тангенциальные координаты астероида составляют $\xi,\eta = 5,5\, arcsec$ без учета атмосферной рефракции. Компоненты скорости астероида $\dot \xi, \dot \eta = 24,17 \,arcsec/hour$. Звезда имеет максимум излучения в спектре на длине волны $\lambda_s = 500\, нм$, астероид - $\lambda_a = 650\, нм$. Условия наблюдений ($P = 556\, мм.рт.ст., t = 3^\circ C$). Построить траектории астероида относительно звезды в тангенциальных координатах с учетом рефракции и без ее учета на протяжении трех часов после начала наблюдений. Оценить, насколько значим эффект атмосферной рефракции для точной астрометрии околоземных астероидов? 